# Gemma-3-4B-it — CNIE Extraction Test
Google 4B, 140 languages, Belebele 59.4 avg. Run Step 1 → restart → run rest.

**Requires HF token** — Gemma is gated. Get token at https://huggingface.co/settings/tokens
then accept license at https://huggingface.co/google/gemma-3-4b-it-qat-q4_0-gguf

In [ ]:
!pip install llama-cpp-python huggingface-hub
print("DONE — Restart runtime, then run next cells.")

In [ ]:
from huggingface_hub import login

# Option A: Colab secrets (recommended)
try:
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
    print("Logged in via Colab secret.")
except:
    # Option B: Paste your token directly
    login(token="hf_YOUR_TOKEN_HERE")  # <-- replace with your token
    print("Logged in via hardcoded token.")

In [ ]:
from llama_cpp import Llama
from huggingface_hub import hf_hub_download
import time, json, re, os

REPO = "google/gemma-3-4b-it-qat-q4_0-gguf"
FILE = "gemma-3-4b-it-q4_0.gguf"
LABEL = "Gemma-3-4B Q4_0_QAT"

print(f"Downloading {LABEL}...")
t0 = time.time()
path = hf_hub_download(repo_id=REPO, filename=FILE)
print(f"Done in {time.time()-t0:.1f}s")

llm = Llama(model_path=path, n_ctx=2048, n_threads=os.cpu_count() or 4, verbose=False)
print(f"Loaded {LABEL}")

In [ ]:
SYSTEM = """You extract personal data from Moroccan CNIE card OCR output.

IGNORE template text: ROYAUME DU MAROC, CARTE NATIONALE D'IDENTITE, المملكة, المغربية, المعربية, البطاقة الوطنية, للتعريف, Né le, مزداد بتاريخ, مزداد بتانيخ, Valable jusqu'au, صالحة الى غاية, المدير العام للأمن الوطني, المدير العام للإمن الوطنى, عبد اللطيف حموشي, عبد اللطيّف حموشي.

FIELDS:
- last_name_fr / last_name_ar: family name (French UPPERCASE)
- first_name_fr / first_name_ar: given name (French UPPERCASE)
- birth_date: DD.MM.YYYY
- birth_place_fr: city UPPERCASE
- birth_place_ar: correct Arabic for the city (translate from French, don't copy OCR)
- expiry_date: DD.MM.YYYY
- card_number: letters + digits
- gender: M or F

RULES: French text is more reliable. Fix OCR errors (1→I, 0→O, 7→T in names). Ignore confidence < 0.5. null for missing fields. ONLY JSON output."""


def parse_json(raw):
    cleaned = re.sub(r'<think>.*?</think>', '', raw, flags=re.DOTALL).strip()
    for s in [cleaned, re.search(r'\{[^{}]*\}', cleaned, re.DOTALL), re.search(r'\{.*\}', cleaned, re.DOTALL)]:
        try:
            txt = s.group() if hasattr(s, 'group') else s
            if txt: return json.loads(txt)
        except: pass
    return {"_raw": raw, "_error": "parse_failed"}


def extract(ocr):
    t0 = time.time()
    r = llm.create_chat_completion(
        messages=[{"role": "user", "content": SYSTEM + "\n\nExtract fields:\n\n" + ocr}],
        max_tokens=512, temperature=0.1)
    elapsed = time.time() - t0
    raw = r["choices"][0]["message"]["content"]
    u = r.get("usage", {})
    print(f"  {elapsed:.1f}s | {u.get('prompt_tokens','?')}→{u.get('completion_tokens','?')} tok")
    return parse_json(raw), round(elapsed, 2)


def score(got, exp):
    correct, lines = 0, []
    for k, v in exp.items():
        g = got.get(k)
        e, gn = (str(v).strip().upper() if v else None), (str(g).strip().upper() if g else None)
        ok = e == gn
        correct += ok
        lines.append(f"  {'OK  ' if ok else 'MISS'} {k}: '{v}' → '{g}'")
    return correct, len(exp), lines

print("Ready.")

In [ ]:
TESTS = {
  "1-Clean": ("""
Arabic OCR:
  - text: 'المملكة المغربية', confidence: 0.95
  - text: 'بطاقة التعريف الوطنية', confidence: 0.92
  - text: 'الشافعي', confidence: 0.88
  - text: 'بلال', confidence: 0.91
French OCR:
  - text: 'ROYAUME DU MAROC', confidence: 0.97
  - text: 'CHAFI', confidence: 0.93
  - text: 'BILAL', confidence: 0.95
  - text: 'Ne le 22.01.2007', confidence: 0.89
  - text: 'a RABAT', confidence: 0.92
  - text: 'Valable jusqu au 19.03.2029', confidence: 0.90
  - text: 'AB123456', confidence: 0.94
  - text: 'M', confidence: 0.96""",
  {"last_name_fr":"CHAFI","first_name_fr":"BILAL","last_name_ar":"الشافعي","first_name_ar":"بلال","birth_date":"22.01.2007","birth_place_fr":"RABAT","birth_place_ar":"الرباط","card_number":"AB123456","expiry_date":"19.03.2029","gender":"M"}),

  "2-Noisy": ("""
Arabic OCR:
  - text: 'الملكة المكربية', confidence: 0.72
  - text: 'لوح', confidence: 0.55
  - text: 'اجا', confidence: 0.60
French OCR:
  - text: 'R0YAUME DU MAR0C', confidence: 0.75
  - text: 'CHAF1', confidence: 0.70
  - text: 'B1LAL', confidence: 0.72
  - text: 'Ne le 22.O1.20O7', confidence: 0.65
  - text: 'a RABA7', confidence: 0.60
  - text: 'Va1ab1e jusqu au 19.03.2029', confidence: 0.58
  - text: 'A8123456', confidence: 0.55""",
  {"last_name_fr":"CHAFI","first_name_fr":"BILAL","birth_date":"22.01.2007","birth_place_fr":"RABAT","birth_place_ar":"الرباط","card_number":"A8123456","expiry_date":"19.03.2029"}),

  "3-Real": ("""
OCR detections:
  - text: 'المعربية', confidence: 0.93
  - text: 'المملكة', confidence: 0.93
  - text: 'ROYAUME DU MAROC', confidence: 0.99
  - text: 'للتعريف', confidence: 0.96
  - text: 'البطاقة الوطنية', confidence: 0.97
  - text: 'CARTE NATIONALE D IDENTITE', confidence: 0.99
  - text: 'بلال', confidence: 0.95
  - text: 'BILAL', confidence: 0.99
  - text: 'شافي', confidence: 0.99
  - text: 'CHAFI', confidence: 0.99
  - text: 'Né le', confidence: 0.87
  - text: '22.01.2001', confidence: 0.92
  - text: 'مزداد بتانيخ', confidence: 0.88
  - text: 'الرياة', confidence: 0.71
  - text: 'a RABAT', confidence: 0.90
  - text: 'Valable jusqu au', confidence: 0.99
  - text: '19.03.2029', confidence: 0.99
  - text: 'صالحة الى غاية', confidence: 0.97
  - text: 'AS13538', confidence: 0.99
  - text: 'M', confidence: 0.99
  - text: 'المدير العام للإمن الوطنى', confidence: 0.97
  - text: 'Thy', confidence: 0.53
  - text: 'عبد اللطيّف حموشي', confidence: 0.91""",
  {"last_name_fr":"CHAFI","first_name_fr":"BILAL","last_name_ar":"شافي","first_name_ar":"بلال","birth_date":"22.01.2001","birth_place_fr":"RABAT","birth_place_ar":"الرباط","card_number":"AS13538","expiry_date":"19.03.2029","gender":"M"}),

  "4-Compound": ("""
OCR detections:
  - text: 'المملكة', confidence: 0.91
  - text: 'المعربية', confidence: 0.89
  - text: 'ROYAUME DU MAROC', confidence: 0.99
  - text: 'البطاقة الوطنية', confidence: 0.95
  - text: 'CARTE NATIONALE D IDENTITE', confidence: 0.99
  - text: 'عبد الرحمن', confidence: 0.87
  - text: 'ABDERRAHMANE', confidence: 0.98
  - text: 'البكاوي', confidence: 0.82
  - text: 'EL BAKAOUI', confidence: 0.97
  - text: 'Né le', confidence: 0.90
  - text: '15.06.1985', confidence: 0.99
  - text: 'الداز البيظاء', confidence: 0.62
  - text: 'a CASABLANCA', confidence: 0.96
  - text: 'Valable jusqu au', confidence: 0.98
  - text: '01.07.2030', confidence: 0.99
  - text: 'BK987654', confidence: 0.99
  - text: 'M', confidence: 0.98
  - text: 'المدير العام للإمن الوطنى', confidence: 0.96
  - text: 'عبد اللطيّف حموشي', confidence: 0.90""",
  {"last_name_fr":"EL BAKAOUI","first_name_fr":"ABDERRAHMANE","last_name_ar":"البكاوي","first_name_ar":"عبد الرحمن","birth_date":"15.06.1985","birth_place_fr":"CASABLANCA","birth_place_ar":"الدار البيضاء","card_number":"BK987654","expiry_date":"01.07.2030","gender":"M"}),

  "5-Female": ("""
OCR detections:
  - text: 'المملكة', confidence: 0.90
  - text: 'المعربية', confidence: 0.88
  - text: 'ROYAUME DU MAROC', confidence: 0.99
  - text: 'للتعريف', confidence: 0.93
  - text: 'البطاقة الوطنية', confidence: 0.96
  - text: 'CARTE NATIONALE D IDENTITE', confidence: 0.99
  - text: 'فاطمة', confidence: 0.91
  - text: 'FATIMA', confidence: 0.99
  - text: 'الزهراء', confidence: 0.78
  - text: 'EZZAHRA', confidence: 0.45
  - text: 'بنيسى', confidence: 0.74
  - text: 'BENNISSI', confidence: 0.98
  - text: 'Né le', confidence: 0.88
  - text: '03.11.1992', confidence: 0.99
  - text: 'وخدة', confidence: 0.55
  - text: 'a OUJDA', confidence: 0.94
  - text: 'Valable jusqu au', confidence: 0.99
  - text: '10.12.2028', confidence: 0.99
  - text: 'CD554433', confidence: 0.99
  - text: 'F', confidence: 0.99
  - text: 'Xr4', confidence: 0.35
  - text: 'المدير العام للإمن الوطنى', confidence: 0.95
  - text: 'عبد اللطيّف حموشي', confidence: 0.89""",
  {"last_name_fr":"BENNISSI","first_name_fr":"FATIMA","last_name_ar":"بنيسى","first_name_ar":"فاطمة","birth_date":"03.11.1992","birth_place_fr":"OUJDA","birth_place_ar":"وجدة","card_number":"CD554433","expiry_date":"10.12.2028","gender":"F"}),

  "6-Minimal": ("""
OCR detections:
  - text: 'ROYAUME DU MAROC', confidence: 0.99
  - text: 'CARTE NATIONALE D IDENTITE', confidence: 0.99
  - text: 'محمد', confidence: 0.93
  - text: 'MOHAMMED', confidence: 0.99
  - text: 'ALAOUI', confidence: 0.98
  - text: '22.08.1970', confidence: 0.97
  - text: 'a FES', confidence: 0.92
  - text: 'EE112233', confidence: 0.99
  - text: 'M', confidence: 0.99""",
  {"last_name_fr":"ALAOUI","first_name_fr":"MOHAMMED","last_name_ar":None,"first_name_ar":"محمد","birth_date":"22.08.1970","birth_place_fr":"FES","birth_place_ar":"فاس","card_number":"EE112233","expiry_date":None,"gender":"M"}),
}

summary = []
for name, (ocr, exp) in TESTS.items():
    print(f"{'='*50}\n{name} — {LABEL}\n{'='*50}")
    fields, t = extract(ocr)
    print(json.dumps(fields, indent=2, ensure_ascii=False))
    c, total, lines = score(fields, exp)
    for l in lines: print(l)
    print(f"  Score: {c}/{total} | {t}s\n")
    summary.append((name, c, total, t))

print(f"{'='*50}\nSUMMARY — {LABEL}\n{'='*50}")
for name, c, t, ts in summary:
    print(f"  {c:2d}/{t:2d} | {ts:5.1f}s | {name}")
tc, tf = sum(s[1] for s in summary), sum(s[2] for s in summary)
print(f"\n  OVERALL: {tc}/{tf} ({100*tc//tf}%) | {sum(s[3] for s in summary):.1f}s total")